In [1]:
import zipfile
import pandas as pd
import ast

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Extract ZIP
# -----------------------------
zip_path = "/content/archive (3).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/tmdb")

# -----------------------------
# Load Dataset
# -----------------------------
movies = pd.read_csv("/content/tmdb/tmdb_5000_movies.csv")
credits = pd.read_csv("/content/tmdb/tmdb_5000_credits.csv")

# -----------------------------
# Merge Datasets
# -----------------------------
movies = movies.merge(credits, on="title")

# -----------------------------
# Select Useful Columns
# -----------------------------
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

# -----------------------------
# Handle Missing Values
# -----------------------------
movies.dropna(inplace=True)

# -----------------------------
# Helper Functions
# -----------------------------
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

# -----------------------------
# Convert Columns
# -----------------------------
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(fetch_director)

# -----------------------------
# Convert Overview to List
# -----------------------------
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# -----------------------------
# Remove Spaces
# -----------------------------
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

# -----------------------------
# Create Tags
# -----------------------------
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

new_df = movies[['movie_id','title','tags']]

new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

# -----------------------------
# Vectorization
# -----------------------------
cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

# -----------------------------
# Similarity Matrix
# -----------------------------
similarity = cosine_similarity(vectors)

# -----------------------------
# Recommendation Function
# -----------------------------
def recommend(movie):

    movie = movie.lower()

    if movie not in new_df['title'].str.lower().values:
        print("Movie not found!")
        return

    index = new_df[new_df['title'].str.lower()==movie].index[0]

    distances = list(enumerate(similarity[index]))

    movies_list = sorted(
        distances,
        reverse=True,
        key=lambda x:x[1]
    )[1:6]

    print("\nRecommended Movies:\n")

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

# -----------------------------
# Example
# -----------------------------
recommend("Avatar")

/tmp/ipykernel_1386/1628099906.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
/tmp/ipykernel_1386/1628099906.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())



Recommended Movies:

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [2]:
movie = "Avatar"

index = new_df[new_df['title'] == movie].index[0]

scores = similarity[index]

print("Similarity Scores:")
print(scores[:10])

Similarity Scores:
[1.         0.08740748 0.05827165 0.03823596 0.17734311 0.11357771
 0.01938917 0.16927779 0.06131393 0.0733674 ]


In [3]:
movie = "Avatar"

index = new_df[new_df['title'] == movie].index[0]

distances = list(enumerate(similarity[index]))

movies_list = sorted(
    distances,
    reverse=True,
    key=lambda x: x[1]
)[1:6]

print("Top 5 Recommended Movies\n")

for i in movies_list:
    print(f"{new_df.iloc[i[0]].title}  --> Similarity Score: {i[1]:.4f}")

Top 5 Recommended Movies

Titan A.E.  --> Similarity Score: 0.2504
Small Soldiers  --> Similarity Score: 0.2478
Independence Day  --> Similarity Score: 0.2428
Ender's Game  --> Similarity Score: 0.2410
Aliens vs Predator: Requiem  --> Similarity Score: 0.2394


In [4]:
print("Similarity Matrix Shape:", similarity.shape)

Similarity Matrix Shape: (4806, 4806)


In [5]:
print("Total Movies:", len(new_df))

Total Movies: 4806


In [6]:
print("Vocabulary Size:", len(cv.get_feature_names_out()))

Vocabulary Size: 5000
